<a href="https://colab.research.google.com/github/Yucheol-Son-BYUI/CSE310_W0_HelloWorld/blob/main/Module5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULL COLAB SETUP (Google Drive -> unzip -> merge -> tf.data)
# Put training1.zip and training2.zip in MyDrive (or adjust paths)
# ============================================================

import os
import zipfile
import shutil
import tensorflow as tf
from google.colab import drive

# ----------------------------
# 0) CONFIG
# ----------------------------
ZIP_PATHS = [
    "/content/drive/MyDrive/training1.zip",
    "/content/drive/MyDrive/training2.zip",
]

EXTRACT_ROOT = "/content/data"        # fast local runtime disk
TRAIN1_DIR = os.path.join(EXTRACT_ROOT, "training1")
TRAIN2_DIR = os.path.join(EXTRACT_ROOT, "training2")
MERGED_DIR = os.path.join(EXTRACT_ROOT, "all")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
VAL_SPLIT = 0.2
SEED = 123

# ----------------------------
# 1) MOUNT DRIVE
# ----------------------------
drive.mount("/content/drive")

# ----------------------------
# 2) VERIFY ZIP FILES EXIST
# ----------------------------
for zp in ZIP_PATHS:
    print("ZIP:", zp, "exists:", os.path.exists(zp))
    if not os.path.exists(zp):
        raise FileNotFoundError(f"Missing zip: {zp}\nUpload it to Drive or fix ZIP_PATHS.")

# ----------------------------
# 3) UNZIP (ONLY IF NEEDED)
# ----------------------------
os.makedirs(EXTRACT_ROOT, exist_ok=True)

need_unzip = not (os.path.exists(TRAIN1_DIR) and os.path.exists(TRAIN2_DIR))
print("Need unzip:", need_unzip)

if need_unzip:
    # optional: clean old extract root
    # shutil.rmtree(EXTRACT_ROOT)
    # os.makedirs(EXTRACT_ROOT, exist_ok=True)

    for zp in ZIP_PATHS:
        print("Extracting:", zp)
        with zipfile.ZipFile(zp, "r") as z:
            z.extractall(EXTRACT_ROOT)

print("Extract root contents:", os.listdir(EXTRACT_ROOT))

# ----------------------------
# 4) MERGE training1 + training2 -> all (ONLY IF NEEDED)
# ----------------------------
os.makedirs(MERGED_DIR, exist_ok=True)

def is_merged_ok():
    # merged ok if it has class folders with at least some images
    if not os.path.exists(MERGED_DIR):
        return False
    classes = [d for d in os.listdir(MERGED_DIR) if os.path.isdir(os.path.join(MERGED_DIR, d))]
    if len(classes) == 0:
        return False
    # check at least one class has images
    for c in classes[:5]:
        if len(os.listdir(os.path.join(MERGED_DIR, c))) > 0:
            return True
    return False

if not is_merged_ok():
    print("Merging into:", MERGED_DIR)

    srcs = [TRAIN1_DIR, TRAIN2_DIR]
    for src in srcs:
        if not os.path.exists(src):
            raise FileNotFoundError(f"Missing extracted folder: {src}\nCheck zip contents/paths.")

        for class_name in os.listdir(src):
            src_class = os.path.join(src, class_name)
            if not os.path.isdir(src_class):
                continue

            dst_class = os.path.join(MERGED_DIR, class_name)
            os.makedirs(dst_class, exist_ok=True)

            for fname in os.listdir(src_class):
                src_file = os.path.join(src_class, fname)
                dst_file = os.path.join(dst_class, fname)

                # avoid overwriting duplicates
                if os.path.exists(dst_file):
                    base, ext = os.path.splitext(fname)
                    i = 1
                    while os.path.exists(os.path.join(dst_class, f"{base}__dup{i}{ext}")):
                        i += 1
                    dst_file = os.path.join(dst_class, f"{base}__dup{i}{ext}")

                shutil.move(src_file, dst_file)

# ----------------------------
# 5) SANITY CHECK
# ----------------------------
classes = [d for d in os.listdir(MERGED_DIR) if os.path.isdir(os.path.join(MERGED_DIR, d))]
print("Merged classes:", len(classes))

def count_images(root):
    total = 0
    for c in os.listdir(root):
        p = os.path.join(root, c)
        if os.path.isdir(p):
            total += len(os.listdir(p))
    return total

print("Total images in merged:", count_images(MERGED_DIR))
print("Example class:", classes[0] if classes else None,
      "files:", (os.listdir(os.path.join(MERGED_DIR, classes[0]))[:10] if classes else None))

if len(classes) == 0:
    raise RuntimeError("MERGED_DIR has no class folders. Check your unzip results and folder structure.")

# ----------------------------
# 6) BUILD DATASETS
# ----------------------------
train_raw = tf.keras.utils.image_dataset_from_directory(
    MERGED_DIR,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_raw = tf.keras.utils.image_dataset_from_directory(
    MERGED_DIR,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_raw.class_names
print("Num classes:", len(class_names))
print("First 20 class names:", class_names[:20], ("..." if len(class_names) > 20 else ""))

# normalize + performance
norm = tf.keras.layers.Rescaling(1./255)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(lambda x, y: (norm(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_raw.map(lambda x, y: (norm(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# show a batch
for x, y in train_ds.take(1):
    print("Batch images:", x.shape, x.dtype, "min/max:", float(tf.reduce_min(x)), float(tf.reduce_max(x)))
    print("Batch labels:", y.shape, "first 10:", y[:10].numpy().tolist())

Mounted at /content/drive
ZIP: /content/drive/MyDrive/training1.zip exists: True
ZIP: /content/drive/MyDrive/training2.zip exists: True
Need unzip: True
Extracting: /content/drive/MyDrive/training1.zip
Extracting: /content/drive/MyDrive/training2.zip
Extract root contents: ['training1', 'training2']
Merging into: /content/data/all
Merged classes: 43
Total images in merged: 39209
Example class: 00001 files: ['00036_00001.jpg', '00047_00015.jpg', '00021_00006.jpg', '00027_00006.jpg', '00042_00001.jpg', '00050_00020.jpg', '00032_00026.jpg', '00019_00024.jpg', '00022_00000.jpg', '00046_00001.jpg']
Found 39209 files belonging to 43 classes.
Using 31368 files for training.
Found 39209 files belonging to 43 classes.
Using 7841 files for validation.
Num classes: 43
First 20 class names: ['00000', '00001', '00002', '00003', '00004', '00005', '00006', '00007', '00008', '00009', '00010', '00011', '00012', '00013', '00014', '00015', '00016', '00017', '00018', '00019'] ...
Batch images: (16, 224, 2

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import mixed_precision

NUM_CLASSES = len(class_names)

mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

augment = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.10, 0.10),
], name="augment")

model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    augment,

    layers.Conv2D(32, 3, activation="leaky_relu", padding="same"),
    layers.Conv2D(32, 3, activation="leaky_relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="swish", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="leaky_relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, activation="relu", padding="same"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="swish"),
    layers.Dense(256, activation="swish"),
    layers.Dropout(.6),
    layers.Dense(256, activation='swish'),
    layers.Dense(128, activation='swish'),

    layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=50, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=150,
    callbacks=callbacks,
)

model.summary()

Mixed precision policy: <DTypePolicy "mixed_float16">
Epoch 1/150
1961/1961 ━━━━━━━━━━━━━━━━━━━━ 128s 55ms/step - accuracy: 0.0809 - loss: 3.3805 - val_accuracy: 0.1478 - val_loss: 3.0054 - learning_rate: 0.0010
Epoch 2/150
1961/1961 ━━━━━━━━━━━━━━━━━━━━ 99s 51ms/step - accuracy: 0.1674 - loss: 2.8956 - val_accuracy: 0.3058 - val_loss: 2.2802 - learning_rate: 0.0010
Epoch 3/150
1961/1961 ━━━━━━━━━━━━━━━━━━━━ 100s 51ms/step - accuracy: 0.3459 - loss: 2.0517 - val_accuracy: 0.5633 - val_loss: 1.2963 - learning_rate: 0.0010
Epoch 4/150
1961/1961 ━━━━━━━━━━━━━━━━━━━━ 99s 50ms/step - accuracy: 0.5553 - loss: 1.3248 - val_accuracy: 0.7262 - val_loss: 0.7939 - learning_rate: 0.0010
Epoch 5/150
1961/1961 ━━━━━━━━━━━━━━━━━━━━ 100s 51ms/step - accuracy: 0.7225 - loss: 0.8372 - val_accuracy: 0.8561 - val_loss: 0.4218 - learning_rate: 0.0010
Epoch 6/150
1961/1961 ━━━━━━━━━━━━━━━━━━━━ 100s 51ms/step - accuracy: 0.8145 - loss: 0.5608 - val_accuracy: 0.9031 - val_loss: 0.3045 - learning_rate: 0.0010


In [ ]:
print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Final train acc:", history.history["accuracy"][-1])
print("Final val acc:", history.history["val_accuracy"][-1])

In [ ]:
import os, hashlib, random

root = "/content/data/all"
seen = set()
dups = 0
checked = 0

for cls in os.listdir(root):
    cls_path = os.path.join(root, cls)
    if not os.path.isdir(cls_path):
        continue
    files = os.listdir(cls_path)
    # sample up to 200 per class to keep it fast
    for f in random.sample(files, min(200, len(files))):
        p = os.path.join(cls_path, f)
        with open(p, "rb") as fp:
            h = hashlib.md5(fp.read()).hexdigest()
        checked += 1
        if h in seen:
            dups += 1
        else:
            seen.add(h)

print("Checked:", checked, " Sample duplicates:", dups)

In [ ]:
model.evaluate(val_ds, verbose=0)

In [ ]:
import numpy as np
import tensorflow as tf

y_true = []
y_pred = []

for x, y in val_ds:
    p = model.predict(x, verbose=0)
    y_true.append(y.numpy())
    y_pred.append(np.argmax(p, axis=1))

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES).numpy()
acc = (y_true == y_pred).mean()
print("Val accuracy (recomputed):", acc)
print("Confusion matrix shape:", cm.shape)

In [ ]:
import numpy as np

wrong = (y_true != y_pred)
print("Num wrong:", int(wrong.sum()), "out of", len(y_true))

# per-class accuracy
per_class = []
for c in range(NUM_CLASSES):
    mask = (y_true == c)
    if mask.sum() == 0:
        per_class.append((c, None, 0))
    else:
        per_class.append((c, float((y_pred[mask] == c).mean()), int(mask.sum())))

# show worst 10 classes (lowest accuracy), ignoring empty classes
per_class_nonempty = [(c, acc, n) for (c, acc, n) in per_class if acc is not None]
worst10 = sorted(per_class_nonempty, key=lambda x: x[1])[:10]
print("Worst 10 classes (class, acc, n):")
for row in worst10:
    print(row)